# Step 11 playground — fluency-aware candidate selection

Both guided loops now take `fluency_weight` (λ): selection maximizes
`cumulative concept score + λ · cumulative token log-probability` instead of concept score alone. λ=0 (default) is the untouched golden path; λ→∞ collapses to unsteered greedy decoding. In between, λ is a knob that trades steering strength for fluency — the E1 Pareto *frontier* becomes a dial.

The candidate token probabilities were already computed inside `_generate_candidates` (the top-k is taken on them) — the blend just stops discarding them, so λ costs nothing.

In [ ]:
import torch

from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

PROMPT = (
    "<|start_header_id|>user<|end_header_id|>Tell me a short story.<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>Once"
)


def answer(text: str) -> str:
    return text.split("assistant<|end_header_id|>")[-1]

## 1. The λ dial

Beam search over the joy+dog mean frame, sweeping λ. Watch the text morph from hard-steered (λ=0) to the model's own greedy story (λ large), with PPL falling along the way.

In [ ]:
joy = fur.get_concept_cached("joy.n.01", 11, 3)
dog = fur.get_concept_cached("dog.n.01", 11, 3)
guide = Concept.average([joy, dog])

unsteered = fur.generate([PROMPT], max_new_tokens=26, do_sample=False)[0]
baseline_ppl = float(fur.loss([answer(unsteered)]).exp())
print(f"[unsteered greedy, ppl={baseline_ppl:.1f}]")
print("   ", answer(unsteered), "\n")

for lam in (0.0, 0.1, 0.2, 0.4, 1.0, 1000.0):
    texts, probe = fur.generate_with_topk_beam_guide(
        [PROMPT], concepts=[guide], k=4, steps=24, fluency_weight=lam
    )
    continuation = answer(texts[0])
    ppl = float(fur.loss([continuation]).exp())
    expression = float(fur.project(continuation, guide).float().mean())
    print(
        f"[lam={lam:g}, ppl={ppl:.1f} ({ppl / baseline_ppl:.1f}x), expression={expression:.3f}]"
    )
    print("   ", continuation, "\n")

## 2. Works in the greedy loop too

Same parameter on the original-style loop (via the multi-guide wrapper — the paper wrapper `generate_with_topk_guide` keeps its fixed golden signature).

In [ ]:
for lam in (0.0, 0.3):
    texts, _ = fur.generate_with_topk_multi_guide(
        [PROMPT], guides=[guide], k=4, steps=24, fluency_weight=lam
    )
    continuation = answer(texts[0])
    ppl = float(fur.loss([continuation]).exp())
    print(f"[greedy loop, lam={lam:g}, ppl={ppl:.1f}]")
    print("   ", continuation, "\n")

## 3. Sanity pins

- λ=0 is byte-identical to the default path in both loops (also pinned by the golden GPU test).
- λ→∞ recovers (approximately) the unsteered greedy text: selection degenerates to pure log-probability.

In [ ]:
base, _ = fur.generate_with_topk_beam_guide([PROMPT], concepts=[guide], k=4, steps=16)
zero, _ = fur.generate_with_topk_beam_guide(
    [PROMPT], concepts=[guide], k=4, steps=16, fluency_weight=0.0
)
print("beam lam=0 == default:", base == zero)

huge, _ = fur.generate_with_topk_beam_guide(
    [PROMPT], concepts=[guide], k=4, steps=24, fluency_weight=1000.0
)
print("\nlam=1000:", answer(huge[0])[:90])
print("greedy:  ", answer(unsteered)[:90])